# 02a · EDA: reports over time, patients, and class balance (E1, E4, M1)
**Owner:** Sayli  |  **Project:** DATA 230 Group 4, FAERS GLP-1

**Data:** `glp1_reports.parquet` from Rutu's `01_data_cleaning.ipynb` (one row = one GLP-1 primary-suspect report).
I only read the shared file. Nothing here re-merges raw FAERS tables.

**Project question:** Which patient, drug and reporting factors are linked to serious outcomes in GLP-1 adverse-event reports?

| Chart | Question it answers | Chart type and why |
|---|---|---|
| **E1** | How have GLP-1 reports changed over time, by drug? | Line chart, one line per drug. Lines are the right encoding for a quantity over time |
| **E4** | Are older patients or one sex more often in serious reports? | (a) split violin of age by outcome and sex: shows the full shape a box plot hides. (b) % serious by age group and sex with 95% intervals and n |
| **M1** | How unbalanced is the target? | (a) bar of serious vs. not serious, count and %. (b) % serious by year, to check the imbalance is stable before pooling years |

Reminder for every line written here: FAERS counts **reports**, not how often side effects happen. We say "among reported cases", never "this drug causes".

In [ ]:
# Setup: put this notebook in the same folder as glp1_reports.parquet,
# or in notebooks/ with the file in data/02_intermediate/. In Colab, upload the file when asked.
import sys
from pathlib import Path

FILE = "glp1_reports.parquet"
IN_COLAB = "google.colab" in sys.modules
SEARCH = [Path.cwd(), Path.cwd() / "data" / "02_intermediate", Path.cwd().parent / "data" / "02_intermediate"]
DATA_PATH = next((d / FILE for d in SEARCH if (d / FILE).exists()), None)

if DATA_PATH is None and IN_COLAB:
    from google.colab import files
    files.upload()
    DATA_PATH = Path.cwd() / FILE
assert DATA_PATH is not None and DATA_PATH.exists(), f"Can't find {FILE}. Put it next to this notebook."
print("Reading:", DATA_PATH)

## 1 · Load the shared file and check it

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

# Shared team rules. Uses src/definitions.py if it is available, otherwise the same values inline.
try:
    sys.path.insert(0, str(Path.cwd().parent)); sys.path.insert(0, str(Path.cwd()))
    from src.definitions import GLP1_DRUGS, DRUG_COLORS, AGE_LABELS
except ImportError:
    GLP1_DRUGS = ["semaglutide", "tirzepatide", "dulaglutide", "liraglutide"]
    DRUG_COLORS = {"semaglutide": "#2563EB", "tirzepatide": "#DC2626",
                   "dulaglutide": "#059669", "liraglutide": "#D97706"}
    AGE_LABELS = ["Under 18", "18-39", "40-64", "65 and over"]

# save to the repo's figures/ and reports/ (one level up when run from notebooks/)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIG_DIR = ROOT / "figures"; FIG_DIR.mkdir(exist_ok=True)
REP_DIR = ROOT / "reports"; REP_DIR.mkdir(exist_ok=True)

reports = pd.read_parquet(DATA_PATH)
print("rows x columns:", reports.shape)

NEEDED = ["primaryid", "year", "quarter", "year_quarter", "drug", "age_years", "age_group", "sex", "is_hcp", "is_serious"]
missing = [c for c in NEEDED if c not in reports.columns]
assert not missing, f"Columns missing: {missing}. Is this Rutu's cleaned file?"
assert reports["primaryid"].is_unique, "Expected one row per report"
assert set(reports["is_serious"].unique()) <= {0, 1}, "is_serious should be 0/1"
assert set(reports["drug"].unique()) <= set(GLP1_DRUGS), "Unexpected drug names"

reports[NEEDED].head()

In [ ]:
# What is actually in the columns I use (Lecture 5, act 1: structure before charts)
df = reports[NEEDED].copy()
print("Quarters:", df["year_quarter"].min(), "to", df["year_quarter"].max(), f"({df['year_quarter'].nunique()} quarters)")
print("\nReports by drug:\n", df["drug"].value_counts().to_string())
print("\nSex:\n", df["sex"].value_counts(dropna=False).to_string())
print("\nAge group:\n", df["age_group"].value_counts(dropna=False).to_string())
print(f"\nAge missing: {df['age_years'].isna().mean():.1%}  |  age_years range: {df['age_years'].min()} to {df['age_years'].max()}")
df["age_years"].describe().round(1)

In [ ]:
# One style for every chart (team rule: 10 x 6 in, 150 dpi, fixed drug colors)
FIGSIZE, DPI = (10, 6), 150      # switch DPI to 500 for the final report (Lecture 3 publication checklist)
plt.rcParams.update({
    "figure.dpi": 100, "savefig.dpi": DPI, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "axes.axisbelow": True,
    "axes.titlesize": 12, "axes.titleweight": "bold",
})
MARKERS = {"semaglutide": "o", "tirzepatide": "s", "dulaglutide": "^", "liraglutide": "D"}
LINES = {"semaglutide": "-", "tirzepatide": "--", "dulaglutide": "-.", "liraglutide": ":"}
SEX_COLORS = {"F": "#7C3AED", "M": "#0E7490"}          # not drug colors, so nobody reads them as drugs
SEX_NAMES = {"F": "Female", "M": "Male"}
AGE_ORDER = AGE_LABELS + ["Unknown"]
GREY, DARK = "#9CA3AF", "#374151"
MIN_N = 30                                              # don't compare groups smaller than this
SAVED, DECISIONS = [], []

try:
    from scipy import stats
except ImportError:
    stats = None
    print("scipy not installed: tests are skipped, charts still run (pip install scipy)")

def wilson(k, n, z=1.96):
    # 95% Wilson interval for a proportion; behaves better than +/- SE for small n
    k, n = np.asarray(k, float), np.asarray(n, float)
    p = k / n
    den = 1 + z**2 / n
    mid = (p + z**2 / (2 * n)) / den
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / den
    return mid - half, mid + half

def cramers_v(table):
    chi2, p, dof, _ = stats.chi2_contingency(table)
    return chi2, p, np.sqrt(chi2 / (table.values.sum() * (min(table.shape) - 1)))

def caption(fig, text, y=-0.02):
    fig.text(0.01, y, text, ha="left", va="top", fontsize=8.5, color=DARK, wrap=True)

def save(fig, name):
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path, bbox_inches="tight")
    SAVED.append(path)
    print("saved ->", path)

def note(insight, decision):
    display(Markdown(f"**Insight:** {insight}\n\n**Decision:** {decision}"))

pretty = lambda d: d.title()

## 2 · E1: How have GLP-1 reports changed over time, by drug?
One line per drug, each with its own color, marker and line style so the chart still reads in grayscale.
The y-axis is a count of reports, so it starts at zero.

In [ ]:
quarters = sorted(df["year_quarter"].unique())
e1 = (df.groupby(["year_quarter", "drug"]).size().unstack(fill_value=0)
        .reindex(index=quarters, columns=GLP1_DRUGS, fill_value=0))
total = e1.sum(axis=1)

# Finding for the title: which drug leads now, and since when has it led without a break
leader = e1.iloc[-1].idxmax()
is_top = e1.idxmax(axis=1).eq(leader)
start = len(is_top)
while start > 0 and is_top.iloc[start - 1]:
    start -= 1
since = quarters[start]
first_q, last_q = quarters[0], quarters[-1]
growth = total.iloc[-1] / total.iloc[0]

if start == 0:
    title = f"{pretty(leader)} has the most GLP-1 reports in every quarter, {first_q} to {last_q}"
elif since == last_q:
    title = f"{pretty(leader)} overtook the other GLP-1 drugs in reports in {last_q}"
else:
    title = f"{pretty(leader)} has had the most GLP-1 reports every quarter since {since}"

fig, ax = plt.subplots(figsize=FIGSIZE)
x = np.arange(len(quarters))
for d in GLP1_DRUGS:
    ax.plot(x, e1[d], color=DRUG_COLORS[d], marker=MARKERS[d], ls=LINES[d], lw=2, ms=5,
            label=f"{pretty(d)} (n={e1[d].sum():,})")
ax.set_xticks(x)
ax.set_xticklabels(quarters, rotation=45, ha="right")
ax.set_xlabel("Quarter FDA received the report")
ax.set_ylabel("Number of reports (GLP-1 as primary suspect)")
ax.set_ylim(0, e1.values.max() * 1.12)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f}"))
ax.set_title(title)
ax.legend(title="Drug", loc="upper left", frameon=False)
caption(fig, f"E1. Primary-suspect GLP-1 reports per quarter, FAERS {first_q}–{last_q}, latest case version only "
             f"(n = {len(df):,} reports). Counts are reports, not usage rates; news coverage and prescribing "
             f"volume both move these lines.", y=-0.07)
save(fig, "E1_trend_by_drug"); plt.show()

e1_table = e1.assign(total=total)
e1_table.loc[[first_q, last_q]]

In [ ]:
share_now = e1.iloc[-1] / total.iloc[-1] * 100
peak_q = total.idxmax()
first_rep = e1.index[e1[leader] > 0][0]

# biggest one-quarter jump for a drug with a real volume (shows how spiky quarterly counts are)
jumps = []
for d in GLP1_DRUGS:
    s_ = e1[d]
    ratio = (s_ / s_.shift(1)).where(s_.shift(1) >= 100)
    if s_.sum() >= 1000 and ratio.notna().any():
        q = ratio.idxmax(); i = quarters.index(q)
        back = i + 1 < len(quarters) and s_.iloc[i + 1] < 0.6 * s_.iloc[i]
        jumps.append((ratio.max(), d, q, back))
jr, jd, jq, jback = max(jumps)
spike_txt = (f" Counts are spiky: {pretty(jd)} jumps {jr:.1f}x in {jq}"
             + (" and falls back the next quarter" if jback else "") + ", which looks like batch reporting or news, not a real change in risk."
             if jr >= 2 else "")

e1_insight = (f"GLP-1 reports went from {total.iloc[0]:,} in {first_q} to {total.iloc[-1]:,} in {last_q} "
              f"({growth:.1f}x)" + ("" if peak_q == last_q else f", peaking in {peak_q} at {total.max():,}") + ". "
              f"{pretty(leader)} first shows up in {first_rep}, leads every quarter since {since}, and makes up "
              f"{share_now[leader]:.0f}% of reports in {last_q}." + spike_txt)
e1_decision = ("Keep time as a feature, but use `year` rather than raw quarter counts because quarterly volume is spiky. "
               "Report volume follows prescribing growth and news coverage, so a spike is not a safety signal on its own; "
               "compare drugs on % serious (E2), not raw counts, and check the target is stable across years before pooling (M1b).")
note(e1_insight, e1_decision)
DECISIONS.append(("year_quarter / year", "keep year as a feature; compare drugs on % serious, not counts",
                  f"reports grew {growth:.1f}x from {first_q} to {last_q}; {pretty(leader)} leads since {since}"))

## 3 · E4: Are older patients or one sex more often in serious reports?
Left panel: split violin of age (years) for serious vs. not serious reports, female on one side and male on the other.
A violin shows the shape a box plot hides (Lecture 5), and `cut=0` stops the density from drawing ages below 0 or above the oldest patient.
Right panel: the share of serious reports in each age group, by sex, with 95% intervals and n, so small groups don't look as certain as big ones.
Reports with unknown sex are left out of the plot but counted in the caption.

In [ ]:
known = df[df["sex"].isin(["F", "M"])].copy()
known["outcome"] = np.where(known["is_serious"] == 1, "Serious", "Not serious")
vio = known.dropna(subset=["age_years"])

# % serious by age group and sex
g = (known.groupby(["age_group", "sex"])["is_serious"].agg(n="size", k="sum").reset_index())
g["pct"] = 100 * g["k"] / g["n"]
lo, hi = wilson(g["k"], g["n"])
g["lo"], g["hi"] = 100 * lo, 100 * hi
g = g[g["n"] >= MIN_N]

# Finding for the title (known age groups only)
by_age = (df[df["age_group"] != "Unknown"].groupby("age_group")["is_serious"].agg(["mean", "size"]))
by_age = by_age[by_age["size"] >= MIN_N].reindex([a for a in AGE_LABELS if a in by_age.index])
top_age, low_age = by_age["mean"].idxmax(), by_age["mean"].idxmin()
by_sex = known.groupby("sex")["is_serious"].mean() * 100
med = vio.groupby("outcome")["age_years"].median()

say = lambda a: "under 18" if a == "Under 18" else "age " + a.replace(" and over", "+").replace("-", "–")
adult = by_age.drop(index="Under 18", errors="ignore")
known_age = g[g["age_group"] != "Unknown"].pivot(index="age_group", columns="sex", values="pct").dropna()
men_higher_all = len(known_age) > 0 and bool((known_age["M"] > known_age["F"]).all())
adult_spread = (adult["mean"].max() - adult["mean"].min()) * 100

if men_higher_all:
    title = (f"Men have a higher share of serious reports than women in every age group "
             f"({by_sex['M']:.1f}% vs. {by_sex['F']:.1f}% overall)")
elif adult_spread >= 5:
    hi_a, lo_a = adult["mean"].idxmax(), adult["mean"].idxmin()
    title = (f"Among adults, {adult.loc[hi_a, 'mean']*100:.1f}% of reports are serious at {say(hi_a)} "
             f"vs. {adult.loc[lo_a, 'mean']*100:.1f}% at {say(lo_a)}")
else:
    title = f"Among adults the serious share barely moves with age ({adult['mean'].min()*100:.1f}%–{adult['mean'].max()*100:.1f}%)"

fig, (a1, a2) = plt.subplots(1, 2, figsize=FIGSIZE, gridspec_kw={"width_ratios": [1, 1.25]})

sns.violinplot(data=vio, x="outcome", y="age_years", hue="sex", hue_order=["F", "M"],
               order=["Not serious", "Serious"], split=True, inner="quart", cut=0,
               palette=SEX_COLORS, linewidth=1, ax=a1)
counts = vio.groupby("outcome").size()
a1.set_xticks([0, 1])
a1.set_xticklabels([f"Not serious\n(n={counts.get('Not serious', 0):,})", f"Serious\n(n={counts.get('Serious', 0):,})"])
a1.set_xlabel("")
a1.set_ylabel("Patient age (years)")
a1.set_ylim(0, 110)
a1.get_legend().remove()
# hatch the male halves so panel a survives grayscale and matches the legend
from matplotlib.collections import PolyCollection
for i, body in enumerate([c for c in a1.collections if isinstance(c, PolyCollection)]):
    if i % 2 == 1:   # seaborn draws F then M for each outcome
        body.set_hatch("//"); body.set_edgecolor("white")
a1.set_title("a · Age by outcome (dashed lines = quartiles)", fontsize=10)

ages = [a for a in AGE_ORDER if a in set(g["age_group"])]
xpos = np.arange(len(ages))
w = 0.38
for i, s in enumerate(["F", "M"]):
    sub = g[g["sex"] == s].set_index("age_group").reindex(ages)
    xs = xpos + (i - 0.5) * w
    a2.bar(xs, sub["pct"], w, color=SEX_COLORS[s], hatch="" if s == "F" else "//",
           edgecolor="white", label=SEX_NAMES[s],
           yerr=[sub["pct"] - sub["lo"], sub["hi"] - sub["pct"]], capsize=3, error_kw={"lw": 1, "ecolor": DARK})
a2.axhline(df["is_serious"].mean() * 100, ls="--", color=GREY, lw=1)
a2.set_xticks(xpos)
nn = g.pivot(index="age_group", columns="sex", values="n").reindex(ages)
a2.set_xticklabels([f"{a.replace(' and over', '+')}\nF {nn.loc[a, 'F']:,.0f}\nM {nn.loc[a, 'M']:,.0f}" for a in ages], fontsize=8)
a2.set_xlabel("Age group (years), with n reports per sex")
a2.set_ylabel("Serious reports (% of reports in the group)")
a2.set_ylim(0, g["hi"].max() * 1.1)
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
fig.legend(handles=[Patch(facecolor=SEX_COLORS["F"], label="Female"),
                    Patch(facecolor=SEX_COLORS["M"], hatch="//", edgecolor="white", label="Male"),
                    Line2D([0], [0], color=GREY, ls="--", label=f"All reports ({df['is_serious'].mean()*100:.1f}%)")],
           loc="upper center", bbox_to_anchor=(0.5, 0.97), ncol=3, frameon=False)
a2.set_title("b · % serious by age group and sex (95% interval)", fontsize=10)

fig.suptitle(title, fontweight="bold", fontsize=12, y=1.02)
n_unknown_sex = (~df["sex"].isin(["F", "M"])).sum()
caption(fig, f"E4. Among reported GLP-1 cases with known sex (n = {len(known):,}; {n_unknown_sex:,} with unknown sex left out). "
             f"Panel a also needs a known age (n = {len(vio):,}). Serious = death, life-threatening, hospitalization, "
             f"disability, congenital anomaly or required intervention. Groups under {MIN_N} reports are not shown.")
fig.tight_layout(rect=(0, 0, 1, 0.91))
save(fig, "E4_age_sex_by_outcome"); plt.show()

In [ ]:
# Is it bigger than chance? (Lecture 5: numeric x categorical -> compare groups; categorical x categorical -> chi-square)
lines = []
if stats is not None:
    s_age = vio.loc[vio["outcome"] == "Serious", "age_years"]
    n_age = vio.loc[vio["outcome"] == "Not serious", "age_years"]
    u, p_age = stats.mannwhitneyu(s_age, n_age)
    _, p_sex, v_sex = cramers_v(pd.crosstab(known["sex"], known["is_serious"]))
    _, p_ag, v_ag = cramers_v(pd.crosstab(df["age_group"], df["is_serious"]))
    lines += [f"Age, serious vs. not (Mann-Whitney): p = {p_age:.2g}",
              f"Sex x serious: chi-square p = {p_sex:.2g}, Cramer's V = {v_sex:.3f}",
              f"Age group x serious: chi-square p = {p_ag:.2g}, Cramer's V = {v_ag:.3f}"]
print("\n".join(lines) if lines else "tests skipped (no scipy)")

unk = df.groupby(df["age_group"].eq("Unknown"))["is_serious"].mean() * 100
print(f"\n% serious, age known: {unk.get(False, np.nan):.1f}%  |  age unknown: {unk.get(True, np.nan):.1f}%")
print(f"Median age: serious {med.get('Serious', np.nan):.0f}, not serious {med.get('Not serious', np.nan):.0f}")
print(f"% serious: men {by_sex.get('M', np.nan):.1f}%, women {by_sex.get('F', np.nan):.1f}%")

In [ ]:
size_word = lambda v: "small" if v < 0.1 else "moderate" if v < 0.3 else "large"
effect = (f" Both links are statistically clear but {size_word(max(v_sex, v_ag))} in size (Cramer's V {v_sex:.2f} for sex, "
          f"{v_ag:.2f} for age group), so neither explains much on its own." if stats is not None else "")

kid = by_age.loc["Under 18"] if "Under 18" in by_age.index else None
infant = df[df["age_group"].eq("Under 18") & (df["age_years"] < 2)]
kid_txt = (f" The exception is under 18: {kid['mean']*100:.1f}% serious, but from only {int(kid['size']):,} reports;"
           f" {len(infant):,} of them are under age 2 ({infant['is_serious'].mean()*100:.0f}% serious), which look like"
           " infant or pregnancy-exposure reports rather than children taking the drug."
           if kid is not None and kid["mean"] > adult["mean"].max() + 0.05 else "")
ka = known_age.drop(index="Under 18", errors="ignore")
sex_txt = (f"Men have the higher share in every age group; among adults that is {ka['M'].min():.1f}%–{ka['M'].max():.1f}% "
           f"for men vs. {ka['F'].min():.1f}%–{ka['F'].max():.1f}% for women." if men_higher_all else
           f"Men {by_sex.get('M', np.nan):.1f}% vs. women {by_sex.get('F', np.nan):.1f}% serious.")

e4_insight = (f"Among reported cases, age barely separates the outcomes: the median age is {med.get('Serious', np.nan):.0f} "
              f"for serious and {med.get('Not serious', np.nan):.0f} for non-serious reports, and adults range only from "
              f"{adult['mean'].min()*100:.1f}% to {adult['mean'].max()*100:.1f}% serious. {sex_txt}" + kid_txt + effect)

# does the unknown-age gap survive once we hold reporter type constant?
unk_by_rep = df.groupby(["is_hcp", df["age_group"].eq("Unknown")])["is_serious"].mean().unstack() * 100
holds = bool((unk_by_rep[True] < unk_by_rep[False] - 1).all() or (unk_by_rep[True] > unk_by_rep[False] + 1).all())
rep_txt = (f" The gap holds for consumer reports ({unk_by_rep.loc[0, True]:.1f}% vs. {unk_by_rep.loc[0, False]:.1f}%) "
           f"and health-professional reports ({unk_by_rep.loc[1, True]:.1f}% vs. {unk_by_rep.loc[1, False]:.1f}%), "
           "so it is not just reporter mix." if holds else "")
gap = abs(unk.get(True, np.nan) - unk.get(False, np.nan))
why_unknown = ("so the missingness itself carries signal" if gap >= 1 else "so dropping them would throw away data for no gain")

e4_decision = ("Keep `sex` and `age_group` as features. Keep 'Under 18' as its own level instead of merging it into 18–39: "
               "it is small but behaves very differently, and the under-2 reports should be checked again before modelling. Keep 'Unknown' as its own level too, and add an age-missing flag: "
               f"age is missing in {df['age_years'].isna().mean():.0%} of reports, and those are serious at "
               f"{unk.get(True, np.nan):.1f}% vs. {unk.get(False, np.nan):.1f}% when age is known, {why_unknown}." + rep_txt)
note(e4_insight, e4_decision)
DECISIONS.append(("sex", "keep as a feature; 'Unknown' stays a category",
                  f"men {by_sex.get('M', np.nan):.1f}% vs. women {by_sex.get('F', np.nan):.1f}% serious"
                  + (", higher in every age group" if men_higher_all else "")))
DECISIONS.append(("age_group", "keep; 'Under 18' stays separate; adults could be one band if the model needs it",
                  f"adults {adult['mean'].min()*100:.1f}%–{adult['mean'].max()*100:.1f}% serious"
                  + (f"; under 18 {kid['mean']*100:.1f}% (n={int(kid['size'])})" if kid is not None else "")))
DECISIONS.append(("age_years missing", "keep rows; 'Unknown' level + age_missing flag; median-impute age_years in the ML pipeline",
                  f"{df['age_years'].isna().mean():.0%} missing; serious {unk.get(True, np.nan):.1f}% vs. {unk.get(False, np.nan):.1f}%"))

## 4 · M1: How unbalanced is the target?
Left: count and share of serious vs. not serious reports. Right: % serious per year with 95% intervals.
If the share is steady across years, we can pool all years for the model and use a stratified split.

In [ ]:
counts = df["is_serious"].value_counts().reindex([0, 1], fill_value=0)
pct = counts / counts.sum() * 100
pos = pct[1]
ratio = counts[0] / counts[1]
naive_acc = 100 - pos

yr = df.groupby("year")["is_serious"].agg(n="size", k="sum")
yr["pct"] = 100 * yr["k"] / yr["n"]
lo, hi = wilson(yr["k"], yr["n"]); yr["lo"], yr["hi"] = 100 * lo, 100 * hi
partial = df.groupby("year")["quarter"].nunique()

title = f"Only {pos:.1f}% of GLP-1 reports are serious: about {ratio:.0f} non-serious for every serious one"

fig, (b1, b2) = plt.subplots(1, 2, figsize=FIGSIZE, gridspec_kw={"width_ratios": [1, 1.3]})
bars = b1.bar(["Not serious\n(is_serious = 0)", "Serious\n(is_serious = 1)"], counts.values,
              color=[GREY, DARK], edgecolor="white", hatch=["", "//"], width=0.6)
for b, c, p in zip(bars, counts.values, pct.values):
    b1.text(b.get_x() + b.get_width() / 2, c + counts.max() * 0.01, f"{c:,}\n({p:.1f}%)",
            ha="center", va="bottom", fontsize=10)
b1.set_ylim(0, counts.max() * 1.18)
b1.set_ylabel("Number of reports")
b1.set_title("a · The target: serious vs. not serious", fontsize=10)
b1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f}"))

xlab = [f"{y}" + ("*" if partial[y] < 4 else "") for y in yr.index]
b2.errorbar(range(len(yr)), yr["pct"], yerr=[yr["pct"] - yr["lo"], yr["hi"] - yr["pct"]],
            fmt="o-", color=DARK, capsize=4, lw=1.5, label="% serious (95% interval)")
for i, (y, r) in enumerate(yr.iterrows()):
    b2.text(i, r["hi"] + 0.4, f"n={int(r['n']):,}", ha="center", va="bottom", fontsize=8)
b2.axhline(pos, ls="--", color=GREY, lw=1, label=f"all years ({pos:.1f}%)")
b2.set_xticks(range(len(yr))); b2.set_xticklabels(xlab)
b2.set_xlim(-0.5, len(yr) - 0.5)
b2.set_xlabel("Year FDA received the report (* = partial year)")
b2.set_ylabel("Serious reports (% of that year's reports)")
b2.set_ylim(0, max(yr["hi"].max() * 1.4, pos * 2))
b2.legend(loc="upper right", frameon=False)
b2.set_title("b · Is the share stable across years?", fontsize=10)

fig.suptitle(title, fontweight="bold", fontsize=12)
caption(fig, f"M1. Class balance of the ML target is_serious among {len(df):,} reported GLP-1 cases. A model that always "
             f"predicts 'not serious' would already be {naive_acc:.1f}% accurate, which is why accuracy is the wrong score here.")
fig.tight_layout()
save(fig, "M1_class_balance"); plt.show()
yr.round(1)

In [ ]:
spread = yr.loc[yr["n"] >= MIN_N, "pct"]
stable = (spread.max() - spread.min()) < 3
m1_insight = (f"Among {len(df):,} reported cases, {counts[1]:,} ({pos:.1f}%) are serious, roughly {ratio:.0f} to 1. "
              f"Always guessing 'not serious' scores {naive_acc:.1f}% accuracy while catching zero serious cases. "
              f"By year the share ranges from {spread.min():.1f}% to {spread.max():.1f}%"
              + (", so it is fairly stable and the 2022–2026 years can be pooled." if stable else ", so it moves enough that year must stay in the model."))
m1_decision = ("Frame the ML task as binary classification (serious vs. not). Handle the imbalance with class weights "
               "first and SMOTE as a comparison, use a stratified train/test split, and judge models on recall, F1 and "
               "PR-AUC for the serious class, not accuracy. Outcome flags (is_death, is_hospitalized, ...) are part of "
               "the target and are never features.")
note(m1_insight, m1_decision)
DECISIONS.append(("is_serious (target)", "binary classification; class weights or SMOTE; stratified split; recall, F1, PR-AUC",
                  f"positives = {pos:.1f}%; always-'not serious' baseline = {naive_acc:.1f}% accuracy"))

## 5 · ML direction (plan only, no model yet)
Text for slide 14, filled from this run.

In [ ]:
ml_text = f'''**ML direction (plan only, results at the final)**
- **Task:** binary classification, serious vs. not serious report (`is_serious`).
- **Why classification:** the project question is about *which* reports end up serious, and the target is a yes/no flag.
- **What EDA tells us:** serious reports are {pos:.1f}% of {len(df):,} cases (about {ratio:.0f}:1). Always predicting "not serious" gives {naive_acc:.1f}% accuracy and is useless.
- **So:** class weights (and SMOTE as a check), stratified split, scored with recall, F1 and PR-AUC, not accuracy.
- **Candidate features from EDA:** drug (E2), reporter type (E6), age group and sex (E4), indication group (E5), reaction group (E3), year (E1).
- **Kept out:** death / hospitalization / life-threatening / disability flags, since they define the target (leakage).'''
display(Markdown(ml_text))

## 6 · Decision table (the output of the EDA)

In [ ]:
dec = pd.DataFrame(DECISIONS, columns=["Column / issue", "Decision", "Because"])
pd.set_option("display.max_colwidth", 160)
display(dec)
dec.to_csv(REP_DIR / "E1_E4_M1_decision_table.csv", index=False)
print("Figures saved this run:")
for p in SAVED:
    print("  ", p)